In [12]:
import json
import os


In [13]:
data_dir = "./data"
intermediate_dir = os.path.join(data_dir, "intermediate")
merged_dir = os.path.join(intermediate_dir, "merged")

processed_dir = os.path.join(data_dir, "processed")
os.makedirs(processed_dir, exist_ok=True)


In [14]:
def load_json(path: str):
	with open(path, encoding="utf-8") as f:
		return json.load(f)


def save_json(path: str, data: dict):
	with open(path, "w", encoding="utf-8") as f:
		json.dump(data, f, ensure_ascii=False, indent=4)


def load_ndjson(path: str):
	lines = []

	with open(path, "r", encoding="utf-8") as f:
		for i, line in enumerate(f):
			line = line.strip()

			if line:
				try:
					lines.append(json.loads(line))
				except json.JSONDecodeError:
					print(f"Invalid JSON in {path} (line {i}): {line[:200]}")

	return lines


def save_ndjson(path: str, lines: list[dict]):
	with open(path, "w", encoding="utf-8") as f:
		for line in lines:
			json.dump(line, f)
			f.write("\n")


In [15]:
def load_tests_users(base_dir: str):
	users = set()

	for phase in ["Pre", "Post"]:
		phase_dir = os.path.join(base_dir, phase)

		for file in os.listdir(phase_dir):
			if file.endswith(".json"):
				path = os.path.join(phase_dir, file)
				content = load_json(path)
				users.update(content.keys())

	return users


In [16]:
def load_traces_users(base_dir: str):
	gameplay_dir = os.path.join(base_dir, "Gameplay")

	users = set()
	for file in os.listdir(gameplay_dir):
		if file.endswith(".ndjson"):
			file_path = os.path.join(gameplay_dir, file)
			content = load_ndjson(file_path)
			for trace in content:
				user = trace.get("actor", {}).get("account", {}).get("name")
				if user:
					users.add(user)	

	return users


In [17]:
def filter_json_users(data: dict, valid_users: set[str]):
	filtered = {}

	for user, answers in data.items():
		if user in valid_users:
			filtered[user] = answers

	return filtered

def filter_trace_users(traces: list[dict], valid_users: set[str]):
	filtered = []

	for trace in traces:
		name = trace.get("actor", {}).get("account", {}).get("name")

		if name in valid_users:
			filtered.append(trace)

	return filtered


In [18]:
test_users = load_tests_users(merged_dir)
trace_users = load_traces_users(merged_dir)

print(f"Users in test: {len(test_users)}")
print(f"Users in gameplay: {len(trace_users)}")

common_users = test_users & trace_users
print(f"Users in both: {len(common_users)}")

only_in_tests = test_users - common_users
only_in_traces = trace_users - common_users

print("\nUsers only in tests:")
print(only_in_tests)

print("\nUsers only in gameplay:")
print(only_in_traces)



Users in test: 113
Users in gameplay: 116
Users in both: 108

Users only in tests:
{'684837b0e48b5a00221a37d0_ryrm', 'memogames', 'qklr', '684837b0e48b5a00221a37d0_naih', 'pzdu'}

Users only in gameplay:
{'ltim', '684837b0e48b5a00221a37d0_wasa', '684837b0e48b5a00221a37d0_sxzv', 'bmqp', '682b4a72c76d2e0023ed4d05_kqud', 'tuxv', 'abxx', 'mlsu'}


In [19]:
excluded_users = {
    "memogames",
    "682b4a72c76d2e0023ed4d05_igbm",
    "682b4a41c76d2e0023ed4b24_dmyj",
    "682b4a41c76d2e0023ed4b24_sxif",
    "682b4a72c76d2e0023ed4d05_agog"
}

# Mantener únicamente los usuarios comunes que no están excluidos
valid_users = common_users - excluded_users

print(f"Usuarios comunes: {len(common_users)}")
print(f"Usuarios excluidos: {len(excluded_users)}")
print(f"Usuarios finales: {len(valid_users)}")


Usuarios comunes: 108
Usuarios excluidos: 5
Usuarios finales: 104


In [ ]:
for folder in os.listdir(intermediate_dir):
    source_dir = os.path.join(intermediate_dir, folder)

    if os.path.isdir(source_dir):
        destination_dir = os.path.join(processed_dir, folder)
        os.makedirs(destination_dir, exist_ok=True)

        # Tests
        for phase in ["Pre", "Post"]:
            source_phase = os.path.join(source_dir, phase)

            if os.path.exists(source_phase):
                destination_phase = os.path.join(destination_dir, phase)
                os.makedirs(destination_phase, exist_ok=True)

                for file in os.listdir(source_phase):
                    if file.endswith(".json"):
                        data = load_json(os.path.join(source_phase, file))
                        filtered = filter_json_users(data, valid_users)

                        path = os.path.join(destination_phase, file)
                        save_json(path, filtered)

        # Gameplay
        source_gameplay = os.path.join(source_dir, "Gameplay")

        if os.path.exists(source_gameplay):
            destination_gameplay = os.path.join(destination_dir, "Gameplay")
            os.makedirs(destination_gameplay, exist_ok=True)

            for file in os.listdir(source_gameplay):
                if file.endswith(".ndjson"):
                    traces = load_ndjson(os.path.join(source_gameplay, file))
                    filtered = filter_trace_users(traces, valid_users)

                    path = os.path.join(destination_gameplay, file)
                    save_ndjson(path, filtered)
